# Yeast Genome Editing

Demonstrates editing a genome sequence using the Python API. We replace a short
subsequence of chromosome VI with a synthetic cassette insert (`ATCGATCG`).

This uses a small synthetic excerpt of the yeast chr VI sequence so the notebook
runs without downloading the full genome. See `edit-yeast-and-export-fasta.md`
for the equivalent CLI workflow using the real genome.

In [ ]:
import os
import tempfile

import gen

tmpdir = tempfile.mkdtemp()
repo = gen.Repository(os.path.join(tmpdir, "gen"))

## Import a reference sequence

We use a short excerpt from the start of yeast chromosome VI (`NC_001138`).
In the full CLI workflow this is the entire 270 kb chromosome imported from
the SGD reference FASTA.

In [ ]:
# First 60 bp of NC_001138 (S288C chr VI)
chr6_excerpt = "GATCTCGCAAGTGCATTCCTAGACTTAATTCATATCTGCTCCTCAACTGTCGATGATGCC"

fasta_path = os.path.join(tmpdir, "chr6.fa")
with open(fasta_path, "w") as f:
    f.write(f">NC_001138\n{chr6_excerpt}\n")

result = repo.import_fasta(fasta_path, name="genome", sample="reference")
print(result)

## Apply an edit

Replace bases 3–5 (0-based, half-open) of `NC_001138` with the cassette
sequence `ATCGATCG`. In the CLI workflow this is done with
`gen update --fasta cassette-edit.fa --start 3 --end 5 --region-name NC_001138`.

`update_with_sequence` lets us pass the replacement inline without writing a file.

In [ ]:
result = repo.update_with_sequence(
    "ATCGATCG",
    sample="reference",
    new_sample="edited",
    region_name="NC_001138",
    start=3,
    end=5,
    name="genome",
)
print(result)

## Export and verify

Export the edited sample to FASTA and confirm `ATCGATCG` appears at position 3.

In [ ]:
fasta_out = os.path.join(tmpdir, "edited.fa")
repo.export_fasta(fasta_out, name="genome", sample="edited")

with open(fasta_out) as f:
    edited = f.read()

print("Reference:", chr6_excerpt)
print("Edited:   ", edited.splitlines()[1])

edited_seq = edited.splitlines()[1]
assert edited_seq[3:11] == "ATCGATCG", f"Edit not found: {edited_seq}"
print("\nEdit verified: ATCGATCG at position 3.")